In [27]:
import os 
import sys
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage, BaseMessage
from langchain.agents import create_agent


In [6]:
load_dotenv()

sys.path.insert(0, os.path.abspath("../Saas-API/"))

llm = ChatOpenAI(model="gpt-5-nano", temperature=0)


## ── TOOLS ──────────────────────────────────────────────────

In [7]:
@tool
def search_documents(query: str, top_k: int=3) -> str:
    """
    Search company documents and knowledge base for relevant information.
    Use this when the user asks about documentation, guides, best practices,
    or any information that might be in the document store.

    Returns the most relevant document passages ranked by relevance.
    """
    try: 
        from app.services.vector_store import query_documents
        results = query_documents(
            query=query,
            top_k=top_k,
            tenant_slug = "acme-corp"
        )
        if not results:
            return f"No relevant documents found for query: {query}"
        
        formatted = []

        for i, r in enumerate(results, 1):
            title = r.get("metadata", {}).get("title", "Untitled")
            score = r.get("score", 0)
            formatted.append(f"[Doc {i} - {title} (relevance: {score:.2f})]\n{r['text']}")
        return "\n\n".join(formatted)
    except ImportError:
        return (f"Document search results for '{query}':\n"
            f"[Mock Result]: PostgreSQL B-tree indexes support equality and "
            f"range queries. Use CREATE INDEX CONCURRENTLY for zero-downtime "
            f"index creation. Composite indexes require leftmost column in WHERE clause."
        )
    except Exception as e:
        return f"Error during document search: {str(e)}"
    
@tool
def query_database(sql_description: str) -> str:
    """
    Query the application database to get counts, statistics, or specific records.
    Describe what data you need in plain English — do NOT write raw SQL.

    Examples of valid queries:
    - "count of all notes in the system"
    - "number of tenants created this week"
    - "count of users per tenant"

    Returns structured data from the database.
    """
    description_lower = sql_description.lower()

    mock_results = {
        "note": "Total notes in system: 47 (across 3 tenants)",
        "tenant": "Active tenants: 3 (acme-corp, demo-tenant, test-org)",
        "user": "Registered users: 12",
        "document": "Indexed documents: 6 (total chunks in vector store: 24)",
    }

    for key, result in mock_results.items():
        if key in description_lower:
            return result
    
    return f"Database query for {sql_description} returned no results. Please refine your query."


@tool
def get_system_health() -> str:
    """
    Check the health status of the application — API, database, cache, and vector store.
    Use this when the user asks about system status, uptime, or service health.
    Returns current status of all services.
    """
    try:
        import httpx
        response = httpx.get("http://localhost:8000/health", timeout=2)
        if response.status_code == 200:
            return f"System Health: {response.json().get('status', 'unknown')}"
        return f"System Health Check Failed: HTTP {response.status_code}"
    except Exception:
        return (
            "System Status:\n"
            "  API: ✅ Running (FastAPI on port 8000)\n"
            "  PostgreSQL: ✅ Connected\n"
            "  Redis: ✅ Connected (cache hit rate: 73%)\n"
            "  ChromaDB: ✅ Connected (3 collections, 24 total chunks)"
        )

tools = [search_documents, query_database, get_system_health]


## ── AGENT SETUP ───────────────────────────────────────────────────────────────

In [15]:
SYSTEM_PROMPT = """You are an intelligent assistant for a multi-tenant SaaS platform.
You help users find information, query data, and understand their system.

You have access to these tools:
- search_documents: Search knowledge base and company documents
- query_database: Get counts and statistics from the database
- get_system_health: Check status of all services

Guidelines:
- Use tools when you need current data — don't guess
- Combine tools when a question needs multiple data sources
- Be concise in your final answer — the user doesn't need to see tool details
- If a tool fails, acknowledge it and answer with what you have
- Always cite which tools you used at the end of your response"""

agent_prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    MessagesPlaceholder(variable_name="chat_history", optional=True),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

agent = create_agent(model=llm, tools=tools, system_prompt=SYSTEM_PROMPT)


## ── MULTI-TURN CHAT ───────────────────────────────────────────────────────

In [21]:
chat_history : list[HumanMessage | AIMessage] = []

def chat(user_input: str) -> dict:
    result = agent.invoke({"messages": chat_history + [HumanMessage(content=user_input)]})["messages"]
    tool_calls = [
        msg.name for msg in result if isinstance(msg, ToolMessage)
    ]

    chat_history.append(HumanMessage(content=user_input))  # Add user input to history
    chat_history.append(result[-1])  # Add the final AIMessage response to history
    return {
        "response": result[-1].content,
        "tool_calls": tool_calls,
        "intermediate_steps": result,
    }

In [36]:
chat_history: list[BaseMessage] = []

def chat(user_input: str) -> dict:
    """
    Single turn of agent conversation.
    Returns answer, tools used, and intermediate steps.
    """
    full_content = ""
    tools_used = []
    iterations = 0
    final_message = None

    print("\nAnswer: ", end="", flush=True)

    for chunk in agent.stream(
        {"messages": chat_history + [HumanMessage(content=user_input)]},
        stream_mode=["messages", "updates"],  # messages = tokens, updates = tool tracking
        version="v2",
    ):
        # ── Token-by-token streaming ──────────────────────────────────────
        if chunk["type"] == "messages":
            token, metadata = chunk["data"]
            node = metadata.get("langgraph_node", "")

            if node == "model":
                for block in token.content_blocks:
                    if block.get("type") == "text" and block.get("text"):
                        print(block["text"], end="", flush=True)  # ← actual streaming
                        full_content += block["text"]

        # ── Step tracking for tools_used and iterations ───────────────────
        elif chunk["type"] == "updates":
            for step_name, step_data in chunk["data"].items():
                iterations += 1
                messages = step_data.get("messages", [])

                if step_name == "tools":
                    for msg in messages:
                        if isinstance(msg, ToolMessage) and msg.name:
                            tools_used.append(msg.name)

                if step_name == "model":
                    for msg in messages:
                        if hasattr(msg, "content") and msg.content:
                            final_message = msg

    print()  # newline after streaming finishes

    if final_message is None:
        final_message = AIMessage(content=full_content)

    # Update history
    chat_history.append(HumanMessage(content=user_input))
    chat_history.append(final_message)

    return {
        "answer": full_content,
        "tools_used": tools_used,
        "iterations": iterations,
    }

## ── DEMO ─────────────────────────────────────────────────────────────────

In [37]:
def main():
    print("=" * 65)
    print("MULTI-TOOL AGENT DEMO")
    print("=" * 65)
    test_questions = [
        # Requires search_documents only
        "What do our documents say about creating PostgreSQL indexes safely?",

        # Requires query_database only
        "How many notes and tenants do we currently have in the system?",

        # Requires both tools + synthesis
        "Give me a summary of our PostgreSQL indexing documentation "
        "and tell me how many documents we've indexed in total.",

        # Requires get_system_health
        "Is everything running okay? Check the system status.",
    ]

    for question in test_questions:
        print(f"\n{'─'*65}")
        print(f"Question: {question}")
        print(f"{'─'*65}")

        result = chat(question)

        # print(f"\nAnswer: {result['answer']}")
        print(f"Tools used: {result['tools_used']}")
        print(f"Iterations: {result['iterations']}")

    print(f"\n{'='*65}")
    print(f"Conversation history: {len(chat_history)} messages")
    print(chat_history)
    print("=" * 65)

if __name__ == "__main__":
    main()

MULTI-TOOL AGENT DEMO

─────────────────────────────────────────────────────────────────
Question: What do our documents say about creating PostgreSQL indexes safely?
─────────────────────────────────────────────────────────────────

Answer: I looked for internal docs about safely creating PostgreSQL indexes, but I didn’t find any relevant results in our knowledge base or document store.

If you’d like, I can broaden the search (e.g., include engineering wikis or external guides) or pull in related documents from other sources. In the meantime, here are general PostgreSQL best practices you can apply safely:

- Use CREATE INDEX CONCURRENTLY for large tables to avoid long locks (note: cannot be run inside a transaction; plan for longer creation time).
- Prefer partial or expression-based indexes to keep size and maintenance low when queries frequently filter on specific predicates.
- After creating an index, run ANALYZE to refresh planner statistics.
- Consider appropriate maintenance_w